In [1]:
import pandas as pd
import glob
from pathlib import Path


def load_all_csvs(input_dir: str) -> pd.DataFrame:
    pattern = f"{input_dir}/*.[cC][sS][vV]"
    files = glob.glob(pattern)
    dfs = []
    for f in files:
        dfs.append(pd.read_csv(f, delimiter=";", low_memory=False))
    if dfs:
        return pd.concat(dfs, ignore_index=True)
    else:
        raise SystemExit(f"Input csv files in {input_dir} not possible to load")


def save(
    data: pd.DataFrame,
    output_path: str,
) -> pd.DataFrame:
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    data.to_csv(output_path, index=False, sep=";")
    print(f"Saved {len(data)} rows to {output_path}")


df = load_all_csvs("./input")

# Convert string DATA_HORA_FATO to datetime
df["DATA_HORA"] = pd.to_datetime(
    df["DATA_HORA_FATO"], format="%d/%m/%Y %H:%M:%S", errors="coerce"
)
df = df.dropna(subset="DATA_HORA", inplace=False)

# Convert string LATITUDE and LONGITUDE to float
df["LATITUDE"] = df["LATITUDE"].astype(str).str.replace(",", ".").astype(float)
df["LONGITUDE"] = df["LONGITUDE"].astype(str).str.replace(",", ".").astype(float)
df = df.dropna(subset=["LATITUDE", "LONGITUDE"])

# Group crimes type
group = df["NATUREZA_FATO"].value_counts()
display(group)

NATUREZA_FATO
ROUBO A TRANSEUNTE                             75220
ROUBO DE VEÍCULO (MOTO)                        20342
ROUBO OUTROS                                   13971
ROUBO DE VEÍCULO (DE PASSEIO)                   7667
ROUBO A CASA COMERCIAL                          6254
ROUBO A TRANSPORTE COLETIVO URBANO              5786
TENTATIVA DE ROUBO                              2736
ROUBO A RESIDÊNCIA                              2274
ROUBO A TRANSPORTE COLETIVO RODOVIÁRIO           977
ROUBO DE VEÍCULO (OUTROS)                        640
ROUBO DE CARGAS                                  636
ROUBO A CORRESPONDENTE BANCÁRIO                  243
EXTORSÃO                                         236
ARROMBAMENTO A BANCO (EXPLOSIVO)                 128
EXTORSÃO MEDIANTE SEQUESTRO                       59
ROUBO A CORREIOS                                  52
ROUBO A BANCO                                     26
ROUBO A CASA LOTÉRICA                             14
ARROMBAMENTO EM AGÊNCIA BANCÁRIA

In [2]:
list_crimes = [
    "ROUBO A TRANSEUNTE",  # x
    # 'ROUBO OUTROS', # x
    "ROUBO A RESIDÊNCIA",
    #
    "ROUBO DE VEÍCULO (MOTO)",
    "ROUBO DE VEÍCULO (DE PASSEIO)",
    "ROUBO DE VEÍCULO (OUTROS)",
    "ROUBO A TRANSPORTE COLETIVO RODOVIÁRIO",
    "ROUBO A TRANSPORTE COLETIVO URBANO",
    "ROUBO A CORRESPONDENTE BANCÁRIO",
    "ROUBO A CASA COMERCIAL",  # x
    "ROUBO A CORREIOS",
]


print("Shape before selecting the specified robbery types: ", df.shape)
df_filtered = df.copy()
# filter by the raw natureza values we care about
df_filtered = df_filtered[df_filtered["NATUREZA_FATO"].isin(list_crimes)]
print("Shape after selecting the specified robbery types: ", df_filtered.shape)


def map_roubo_group(natureza):
    s = str(natureza).upper()

    # Use explicit exact-match dictionary first, then substring-based mapping.
    exact_map = {
        # street
        "ROUBO A TRANSEUNTE": "street_robbery",
        # vehicle
        "ROUBO DE VEÍCULO (MOTO)": "vehicle_robbery",
        "ROUBO DE VEÍCULO (DE PASSEIO)": "vehicle_robbery",
        "ROUBO DE VEÍCULO (OUTROS)": "vehicle_robbery",
        # public transport
        "ROUBO A TRANSPORTE COLETIVO RODOVIÁRIO": "public_transport_robbery",
        "ROUBO A TRANSPORTE COLETIVO URBANO": "public_transport_robbery",
        # commercial
        "ROUBO A CORRESPONDENTE BANCÁRIO": "commercial_robbery",
        "ROUBO A CASA COMERCIAL": "commercial_robbery",
        "ROUBO A CORREIOS": "commercial_robbery",
        # residential
        "ROUBO A RESIDÊNCIA": "residential_robbery",
    }
    if s in exact_map:
        return exact_map[s]

    return None


# apply mapping and keep only rows mapped to one of our groups
df_filtered["ROUBO_GROUP"] = df_filtered["NATUREZA_FATO"].apply(map_roubo_group)
df_filtered = df_filtered[df_filtered["ROUBO_GROUP"].notna()].copy()
print(
    "Shape after grouping and filtering (keeps only mapped groups):", df_filtered.shape
)

# normalise original text column for readability (optional)
df_filtered["NATUREZA_FATO"] = df_filtered["NATUREZA_FATO"].str.title()

Shape before selecting the specified robbery types:  (137300, 30)
Shape after selecting the specified robbery types:  (119455, 30)
Shape after grouping and filtering (keeps only mapped groups): (119455, 31)


In [3]:
# Select only the desired columns
df_with_selected_columns = df_filtered[
    [
        "LONGITUDE",
        "LATITUDE",
        "ROUBO_GROUP",
        "DATA_HORA",
        "CIDADE_FATO",
    ]
]


df_arapiraca = df_with_selected_columns.query("CIDADE_FATO == 'Arapiraca'").reset_index(
    drop=True
)
save(df_arapiraca, output_path="./output/arapiraca/filtered_arapiraca.csv")

df_maceio = df_with_selected_columns.query("CIDADE_FATO == 'Maceió'").reset_index(
    drop=True
)
save(df_maceio, output_path="./output/maceio/filtered_maceio.csv")

Saved 17421 rows to ./output/arapiraca/filtered_arapiraca.csv
Saved 72787 rows to ./output/maceio/filtered_maceio.csv
